# Vanilla vs inter-block decoder stacks

This notebook explains how the two trunk modes in `NanoLLM` differ.

- **Conceptual primer (no code):** in [vanilla_vs_inter_block_decoder.md](./vanilla_vs_inter_block_decoder.md), read **“Paper intuition: Attention Residuals and Block AttnRes”** — normal residuals vs depth-wise mixing, scalar toy (5.3 vs 10), block summaries (6.2), and how that maps to this repo.
- **Slow walkthrough (start here if confused):** tiny shapes (`B=1, T=4, D=8`), lots of `print()` — you see `blocks`, `partial`, depth softmax weights, and macro-block boundaries.
- **Later sections:** full `build_model` forward passes and parameter counts.

- **Vanilla** (`block_attn_residuals=False`): GPT-style `DecoderBlock`.
- **Inter-block** (`block_attn_residuals=True`): `InterBlockAttnDecoderBlock` + `block_attn_res`.

Static reference: [vanilla_vs_inter_block_decoder.md](./vanilla_vs_inter_block_decoder.md).

## 0) Setup

Run from repo root or `docs/`. Requires PyTorch.

In [6]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")
sys.path.insert(0, str(repo_root / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F

from nano_llm.model import build_model

## Slow walkthrough — printed tensors

The code below mirrors `src/nano_llm/layers/block_attn_residual.py` and the loop in `NanoLLM._run_inter_block_decoder_stack`, with **prints** so you can read what happens.

**Vocabulary:**
- **`blocks`**: list of `[B,T,D]` snapshots (starts as `[x_emb]`).
- **`partial`**: inside a macro-block, carries $\Delta_{attn}+\Delta_{mlp}$ so far; `None` at macro-block start.
- **Depth mixing**: stack those tensors along a new axis `n`, softmax **over `n`**, weighted sum → new `[B,T,D]` that feeds Attn or FFN.

### A) `block_attn_res` only — softmax over depth, not over time

Each **depth slot** is a full sequence tensor `[B,T,D]`. At each position `(b,t)` we get one weight per slot; weights sum to 1 across slots.

In [7]:
from nano_llm.layers.block_attn_residual import RMSNorm


def verbose_block_attn_res(
    blocks: list[torch.Tensor],
    partial: torch.Tensor | None,
    proj: nn.Linear,
    norm: RMSNorm,
    label: str,
    *,
    quiet: bool = False,
) -> torch.Tensor:
    """Same math as nano_llm.layers.block_attn_residual.block_attn_res, with optional prints."""
    if partial is None:
        v = torch.stack(blocks, dim=0)
        names = [f"blocks[{i}]" for i in range(len(blocks))]
    else:
        v = torch.stack(blocks + [partial], dim=0)
        names = [f"blocks[{i}]" for i in range(len(blocks))] + ["partial (intra–macro-block sum)"]
    n, bsz, t, d = v.shape
    if not quiet:
        print(f"    [{label}]")
        print(f"      Stack depth n={n}: " + " | ".join(names))
        print(f"      Tensor shape [n,B,T,D] = [{n}, {bsz}, {t}, {d}]")

    flat = v.reshape(n * bsz * t, d)
    k = norm(flat).reshape(n, bsz, t, d)
    w = proj.weight.squeeze(0)
    logits = torch.einsum("d,nbtd->nbt", w, k)
    alpha = F.softmax(logits, dim=0)
    out = torch.einsum("nbt,nbtd->btd", alpha, v)

    if not quiet:
        bi, ti = 0, 0
        print(f"      Depth logits @ (b={bi},t={ti}): {[round(x, 4) for x in logits[:, bi, ti].tolist()]}")
        print(f"      Softmax α over depth @ (0,0):     {[round(x, 4) for x in alpha[:, bi, ti].tolist()]}")
        for i in range(n):
            bar = "█" * max(1, int(alpha[i, bi, ti].item() * 36))
            print(f"      α[{i}] {names[i][:28]:28s} {bar} {alpha[i, bi, ti].item():.3f}")
    return out


torch.manual_seed(123)
D = 8
proj = nn.Linear(D, 1, bias=False)
nn.init.normal_(proj.weight)
norm = RMSNorm(D)

# Two easy-to-read blocks: block0 lights dim0, block1 lights dim1
b0 = torch.zeros(1, 4, D)
b0[..., 0] = 1.0
b1 = torch.zeros(1, 4, D)
b1[..., 1] = 1.0

print("=== Demo: mix only [blocks[0], blocks[1]] (start of macro-block) ===")
out0 = verbose_block_attn_res([b0, b1], None, proj, norm, "input to self-attention")
print(f"    Mixed h[0,0,:2] = {out0[0, 0, :2].tolist()}  (blend of dim0 and dim1 based on α)\n")

fake_partial = torch.zeros(1, 4, D)
fake_partial[..., 2] = 0.5
print("=== Demo: mix [blocks[0], blocks[1], partial] (inside macro-block) ===")
_ = verbose_block_attn_res([b0, b1], fake_partial, proj, norm, "input to self-attention")

=== Demo: mix only [blocks[0], blocks[1]] (start of macro-block) ===
    [input to self-attention]
      Stack depth n=2: blocks[0] | blocks[1]
      Tensor shape [n,B,T,D] = [2, 1, 4, 8]
      Depth logits @ (b=0,t=0): [-3.3854, 0.5919]
      Softmax α over depth @ (0,0):     [0.0184, 0.9816]
      α[0] blocks[0]                    █ 0.018
      α[1] blocks[1]                    ███████████████████████████████████ 0.982
    Mixed h[0,0,:2] = [0.018391568213701248, 0.9816084504127502]  (blend of dim0 and dim1 based on α)

=== Demo: mix [blocks[0], blocks[1], partial] (inside macro-block) ===
    [input to self-attention]
      Stack depth n=3: blocks[0] | blocks[1] | partial (intra–macro-block sum)
      Tensor shape [n,B,T,D] = [3, 1, 4, 8]
      Depth logits @ (b=0,t=0): [-3.3854, 0.5919, -2.7502]
      Softmax α over depth @ (0,0):     [0.0178, 0.9487, 0.0335]
      α[0] blocks[0]                    █ 0.018
      α[1] blocks[1]                    ██████████████████████████████████ 0

### B) One `InterBlockAttnDecoderBlock` forward (matches library code)

Steps:
1. Build `h_a` = mix(`blocks`, `partial?`) → LN → causal Attn → `Δ_attn`.
2. Build `h_m` = mix(`blocks`, `Δ_attn`) → LN → FFN → `Δ_mlp`.
3. `x_out = x + Δ_attn + Δ_mlp` (both residuals from **this layer’s** `x`).
4. Update `partial` or append `x_out` to `blocks` when a macro-block ends.

In [8]:
from nano_llm.layers.block_attn_residual import InterBlockAttnDecoderBlock


def verbose_inter_block_layer(
    layer: InterBlockAttnDecoderBlock,
    x: torch.Tensor,
    blocks: list[torch.Tensor],
    partial: torch.Tensor | None,
    layer_index: int,
    macro_block_size: int,
    *,
    quiet: bool = False,
) -> tuple[torch.Tensor, list[torch.Tensor], torch.Tensor | None]:
    start_of_macro = layer_index % macro_block_size == 0
    partial_for_v = None if start_of_macro else partial

    if not quiet:
        print(f"\n>>> Layer {layer_index}  |  start_of_macro_block = {start_of_macro}")
        print(f"    len(blocks)={len(blocks)}  |  partial is None? {partial is None}")
        if partial_for_v is None:
            print("    Attn-path mix uses: blocks ONLY (partial excluded)")
        else:
            print("    Attn-path mix uses: blocks + partial")

    h_a = verbose_block_attn_res(
        blocks,
        partial_for_v,
        layer.attn_res_proj,
        layer.attn_res_norm,
        "mix → self-attention",
        quiet=quiet,
    )
    with torch.no_grad():
        delta_a = layer.attn(layer.ln1(h_a))
    partial_after_attn = delta_a
    if not quiet:
        print(f"    Δ_attn shape {tuple(delta_a.shape)}  (raw attn output; not yet added to x)")

    h_m = verbose_block_attn_res(
        blocks,
        partial_after_attn,
        layer.mlp_res_proj,
        layer.mlp_res_norm,
        "mix → FFN (last depth slot = Δ_attn)",
        quiet=quiet,
    )
    with torch.no_grad():
        delta_m = layer.ffn(layer.ln2(h_m))
    if not quiet:
        print(f"    Δ_mlp shape {tuple(delta_m.shape)}")

    x_out = x + delta_a + delta_m
    partial_out = partial_after_attn + delta_m
    if not quiet:
        print("    x_out = x + Δ_attn + Δ_mlp   (parallel residuals from same x)")
        print(f"    x mean {x.mean().item():.4f}  →  x_out mean {x_out.mean().item():.4f}")

    if (layer_index + 1) % macro_block_size == 0:
        blocks_out = blocks + [x_out]
        partial_next = None
        if not quiet:
            print(f"    *** End of macro-block: append snapshot. len(blocks) now {len(blocks_out)} ***")
    else:
        blocks_out = blocks
        partial_next = partial_out
        if not quiet:
            print(f"    Inside macro-block: partial_next = Δ_attn + Δ_mlp (shape {tuple(partial_next.shape)})")

    return x_out, blocks_out, partial_next

### C) Four layers, `macro_block_size=2` — watch `blocks` and `partial`

Same schedule as `NanoLLM._run_inter_block_decoder_stack`: after layers 1 and 3 a new snapshot is appended (0-based indices).

In [9]:
from nano_llm.layers.block_attn_residual import trim_blocks

torch.manual_seed(0)
B, T, D, FF = 1, 4, 8, 32
macro_block_size = 2
max_block_repr = 9

x_emb = torch.randn(B, T, D)
blocks: list[torch.Tensor] = [x_emb.clone()]
partial: torch.Tensor | None = None
x = x_emb.clone()

layers_ib = [InterBlockAttnDecoderBlock(D, 2, FF, dropout=0.0) for _ in range(4)]

print("=" * 70)
print("INTER-BLOCK STACK (4 layers, macro_block_size=2)")
print("=" * 70)

with torch.no_grad():
    for li, layer in enumerate(layers_ib):
        x, blocks, partial = verbose_inter_block_layer(
            layer, x, blocks, partial, li, macro_block_size
        )
        before = len(blocks)
        blocks = trim_blocks(blocks, max_block_repr)
        if len(blocks) != before:
            print(f"    trim_blocks: {before} → {len(blocks)} (cap={max_block_repr})")

print("\nDone. Final x shape:", tuple(x.shape))

INTER-BLOCK STACK (4 layers, macro_block_size=2)

>>> Layer 0  |  start_of_macro_block = True
    len(blocks)=1  |  partial is None? True
    Attn-path mix uses: blocks ONLY (partial excluded)
    [mix → self-attention]
      Stack depth n=1: blocks[0]
      Tensor shape [n,B,T,D] = [1, 1, 4, 8]
      Depth logits @ (b=0,t=0): [0.7773]
      Softmax α over depth @ (0,0):     [1.0]
      α[0] blocks[0]                    ████████████████████████████████████ 1.000
    Δ_attn shape (1, 4, 8)  (raw attn output; not yet added to x)
    [mix → FFN (last depth slot = Δ_attn)]
      Stack depth n=2: blocks[0] | partial (intra–macro-block sum)
      Tensor shape [n,B,T,D] = [2, 1, 4, 8]
      Depth logits @ (b=0,t=0): [0.6247, -0.3565]
      Softmax α over depth @ (0,0):     [0.7273, 0.2727]
      α[0] blocks[0]                    ██████████████████████████ 0.727
      α[1] partial (intra–macro-block s █████████ 0.273
    Δ_mlp shape (1, 4, 8)
    x_out = x + Δ_attn + Δ_mlp   (parallel residual

### D) Vanilla four layers — only one tensor `x`

No `blocks` / `partial`. Each `DecoderBlock` does `x = x + Attn(LN(x))` then `x = x + FFN(LN(x))` — the FFN’s norm sees **`x` after the first residual**.

In [10]:
from nano_llm.layers.decoder_block import DecoderBlock

B, T, D, FF = 1, 4, 8, 32
torch.manual_seed(0)
x_emb = torch.randn(B, T, D)
x = x_emb.clone()
layers_v = [DecoderBlock(D, 2, FF, dropout=0.0) for _ in range(4)]

print("=" * 70)
print("VANILLA STACK (4 layers)")
print("=" * 70)

with torch.no_grad():
    for i, block in enumerate(layers_v):
        x_in = x
        x = block(x)
        print(f"\nLayer {i}: x shape {tuple(x.shape)}  |  mean {x_in.mean().item():.4f} → {x.mean().item():.4f}")

print("\nNo blocks list; single stream only.")

VANILLA STACK (4 layers)

Layer 0: x shape (1, 4, 8)  |  mean 0.0066 → -0.1086

Layer 1: x shape (1, 4, 8)  |  mean -0.1086 → -0.1981

Layer 2: x shape (1, 4, 8)  |  mean -0.1981 → -0.1708

Layer 3: x shape (1, 4, 8)  |  mean -0.1708 → -0.2346

No blocks list; single stream only.


### E) Sanity check: library forward matches our verbose layer

Same weights, same inputs → `verbose_inter_block_layer` and `InterBlockAttnDecoderBlock.forward` should return the same `x_out` and state.

In [11]:
from nano_llm.layers.block_attn_residual import block_attn_res as block_attn_res_lib

B, T, D, FF = 1, 4, 8, 32  # same as section C (so this cell runs standalone)

torch.manual_seed(7)
proj_s = nn.Linear(D, 1, bias=False)
nn.init.normal_(proj_s.weight)
norm_s = RMSNorm(D)
bls = [torch.randn(1, 4, D), torch.randn(1, 4, D)]
prt = torch.randn(1, 4, D)
v_out = verbose_block_attn_res(bls, prt, proj_s, norm_s, "sanity", quiet=True)
l_out = block_attn_res_lib(bls, prt, proj_s, norm_s)
mix_diff = (v_out - l_out).abs().max().item()
print("block_attn_res: verbose vs library max diff:", mix_diff)

torch.manual_seed(42)
layer = InterBlockAttnDecoderBlock(D, 2, FF, dropout=0.0)
layer.eval()
xv = torch.randn(B, T, D)
bl = [torch.randn(B, T, D)]
pt: torch.Tensor | None = None
li, mb = 0, 2

with torch.no_grad():
    x_v, blocks_v, p_v = verbose_inter_block_layer(
        layer, xv.clone(), [t.clone() for t in bl], pt, li, mb, quiet=True
    )
with torch.no_grad():
    x_l, blocks_l, p_l = layer(xv, bl, pt, layer_index=li, macro_block_size=mb)

print("InterBlockAttnDecoderBlock: verbose vs library max diff:", (x_v - x_l).abs().max().item())
print("len(blocks):", len(blocks_v), len(blocks_l), "| partial all None?", p_v is None, p_l is None)

block_attn_res: verbose vs library max diff: 0.0
InterBlockAttnDecoderBlock: verbose vs library max diff: 0.0
len(blocks): 1 1 | partial all None? False False


---

## 1) Vanilla stack (full `NanoLLM`)

Each layer:

1. `x = x + Attn(LN(x))`
2. `x = x + FFN(LN(x))` — the FFN sees the **output** of step 1, not the original `x`.

No extra list of tensors is carried between layers.

In [12]:
vocab_size = 256
batch, seq = 2, 32
tok = torch.randint(0, vocab_size, (batch, seq))

model_v = build_model(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    num_layers=4,
    d_ff=256,
    max_len=seq,
    dropout=0.0,
    weight_tie=True,
    position_encoding="sinusoidal",
    block_attn_residuals=False,
)
model_v.eval()
print("decoder_stack:", model_v.decoder_stack)
with torch.no_grad():
    logits_v = model_v(tok)
print("logits shape:", tuple(logits_v.shape))

decoder_stack: vanilla
logits shape: (2, 32, 256)


## 2) Inter-block stack (full `NanoLLM`)

Same attention and FFN **modules**, but the trunk uses the inter-block loop and `InterBlockAttnDecoderBlock`.

In [13]:
model_ib = build_model(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    num_layers=4,
    d_ff=256,
    max_len=seq,
    dropout=0.0,
    weight_tie=True,
    position_encoding="sinusoidal",
    block_attn_residuals=True,
    macro_block_size=2,
    max_block_representations=9,
)
model_ib.eval()
print("decoder_stack:", model_ib.decoder_stack)
with torch.no_grad():
    logits_ib = model_ib(tok)
print("logits shape:", tuple(logits_ib.shape))

decoder_stack: inter_block
logits shape: (2, 32, 256)


## 3) Same input, different trunk — logits differ

Architectures differ, so **do not** expect equal logits. Both maps are `[B, T, vocab_size]`.

In [14]:
with torch.no_grad():
    diff = (logits_v - logits_ib).abs().mean().item()
print(f"mean |logits_vanilla - logits_inter_block| = {diff:.6f}")

mean |logits_vanilla - logits_inter_block| = 9.186075


## 4) Parameter count (tiny model)

Inter-block adds small per-layer heads (`Linear(d_model, 1)` + `RMSNorm`) for the two `block_attn_res` paths, so parameter count is **slightly higher** at the same width/depth.

In [15]:
def n_params(m: torch.nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

nv, nib = n_params(model_v), n_params(model_ib)
print(f"vanilla      params: {nv:,}")
print(f"inter-block  params: {nib:,}")
print(f"delta        params: {nib - nv:,}")

vanilla      params: 216,448
inter-block  params: 217,472
delta        params: 1,024


## 5) Training / checkpoints

| Mode | CLI |
|------|-----|
| Vanilla | omit `--block-attn-residuals` |
| Inter-block | pass `--block-attn-residuals` |

Checkpoint `config` must match at load time (`block_attn_residuals`, `macro_block_size`, `max_block_representations`). Swapping trunk type without retraining will not load correctly.

## 6) Optional: RoPE

Both stacks support `position_encoding="rope"` the same way; only the **block class** and **forward loop** change. Try setting `position_encoding="rope"` and `max_len >= seq` in `build_model` if you want to mirror production configs.